# Relative Active Graph — C4 Training Notebook

Train the **MTLG lexicon** and **TRD bootstrapper** on a streaming subset of the English [C4 corpus](https://huggingface.co/datasets/allenai/c4) from HuggingFace, then demonstrate inference on new sentences and launch the **Streamlit UI** for interactive exploration.

### Pipeline
1. **Clone** the RAG repo from GitHub (source of truth)
2. **Stream** C4 (English, ~500 sentences) — no full download
3. **Parse** → Universal Dependencies → MTLG modal graphs
4. **Induce** MTLG lexicon (MLE) + **Bootstrap** TRD clusters (k-means)
5. **Evaluate** + **Visualise** distributions
6. **Inference demo** — analyse new sentences
7. **Launch** interactive Streamlit UI


## 1 · Install Dependencies

In [ ]:
%%capture
!pip install datasets stanza numpy scipy networkx sentencepiece matplotlib tqdm streamlit pyngrok
print("Dependencies installed.")

## 2 · Clone Repository from GitHub

We pull the source modules directly from the canonical repo — no inline copies needed.
This ensures the notebook always reflects the **current codebase**.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Eupham/Relative_Active_Graph.git"
REPO_DIR = "Relative_Active_Graph"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only
else:
    print(f"Cloning {REPO_URL} ...")
    !git clone --depth 1 {REPO_URL}

# Add induction pipeline to Python path
INDUCTION_PATH = os.path.join(REPO_DIR, "lcs", "induction")
REPO_ROOT = os.path.abspath(REPO_DIR)
for p in [INDUCTION_PATH, REPO_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Repository ready at: {REPO_ROOT}")
print(f"Induction path: {INDUCTION_PATH}")

## 3 · Demo Configuration

In [ ]:
# ── Tune these for a faster / richer demo ───────────────────────────────────
LANGUAGE       = "en"   # ISO 639-1  (any mC4 language)
TRAIN_SAMPLES  = 500    # sentences used for lexicon + TRD induction
HELD_OUT       = 50     # sentences reserved for evaluation
N_TRD_CLUSTERS = 16     # TRD k-means clusters  (16 ≈ fast demo; try 64 for full run)
LEXICON_PATH   = f"{LANGUAGE}_lexicon.json"
TRD_PATH       = f"{LANGUAGE}_trds.json"
# ─────────────────────────────────────────────────────────────────────────────
print(f"Config: lang={LANGUAGE}, train={TRAIN_SAMPLES}, held_out={HELD_OUT}, clusters={N_TRD_CLUSTERS}")

## 4 · Download Stanza Language Model

In [ ]:
import stanza, logging
logging.basicConfig(level=logging.WARNING)   # suppress verbose output
stanza.download(LANGUAGE, verbose=False)
print(f"Stanza '{LANGUAGE}' model ready.")

## 5 · Stream C4 Data

We use HuggingFace `datasets` in **streaming mode** — only the sentences we need are downloaded, not the entire corpus.

In [ ]:
import sys, time

from mc4_stream import stream_mc4

N_TOTAL = TRAIN_SAMPLES + HELD_OUT
print(f"Streaming {N_TOTAL} sentences from allenai/c4 ({LANGUAGE}) …")

t0 = time.time()
raw_sentences = []
for item in stream_mc4(LANGUAGE, max_samples=N_TOTAL):
    raw_sentences.append(item["text"])
    if len(raw_sentences) % 100 == 0:
        print(f"  {len(raw_sentences)}/{N_TOTAL} …")

train_sentences = raw_sentences[:TRAIN_SAMPLES]
held_out_sentences = raw_sentences[TRAIN_SAMPLES:]
print(f"Done in {time.time()-t0:.1f}s — {len(train_sentences)} train, {len(held_out_sentences)} held-out.")

## 6 · Parse & Build MTLG Graphs

Each sentence goes through:
- **Morphological preprocessing** (FST suffix stripping for agglutinative languages)
- **UD parsing** via Stanza
- **MTLG conversion** — dependency arcs become typed modal edges (◇ / □ / ◊)

In [ ]:
from ud_parser import UdParser
from ud_to_mtlg import ud_tree_to_mtlg
from morphological_fst import preprocess_for_type_assignment

parser = UdParser(LANGUAGE)

def sentences_to_graphs(sentences):
    graphs = []
    for sent in sentences:
        try:
            tokens = preprocess_for_type_assignment(sent, LANGUAGE)
            clean  = " ".join(t.split("[")[0] for t in tokens)
            tree   = parser.parse(clean)
            g      = ud_tree_to_mtlg(tree)
            if g.nodes:
                graphs.append(g)
        except Exception as e:
            pass   # skip malformed sentences
    return graphs

print("Parsing training sentences …")
t0 = time.time()
train_graphs    = sentences_to_graphs(train_sentences)
held_out_graphs = sentences_to_graphs(held_out_sentences)
print(f"Done in {time.time()-t0:.1f}s — {len(train_graphs)} train graphs, {len(held_out_graphs)} held-out.")

## 7 · Induce MTLG Lexicon

The `MtlgInducer` observes each MTLG graph and accumulates `(lemma, modal_mode, ucca_cat, arity)` counts, then normalises to log-probabilities (MLE).

In [ ]:
from pathlib import Path
from mtlg_inducer import MtlgInducer

print("Inducing MTLG lexicon …")
t0 = time.time()
inducer = MtlgInducer(LANGUAGE)
lexicon = inducer.induce_from_stream(iter(train_graphs), max_trees=len(train_graphs))
lexicon.save(LEXICON_PATH)
print(f"Lexicon: {len(lexicon.entries)} lemmas — saved to {LEXICON_PATH}  ({time.time()-t0:.1f}s)")

## 8 · Bootstrap TRD Clusters

Each MTLG graph is projected to a **modal type profile vector** (normalised histogram over `(mode, ucca_cat)` pairs).  k-means clusters these vectors into TRD centroids.

In [ ]:
from trd_bootstrap import TrdBootstrapper

print(f"Bootstrapping {N_TRD_CLUSTERS} TRD clusters …")
t0 = time.time()
bootstrapper = TrdBootstrapper(n_clusters=N_TRD_CLUSTERS)
trds = bootstrapper.bootstrap(train_graphs, LANGUAGE)
import json as _json
with open(TRD_PATH, "w") as fh:
    _json.dump([t.__dict__ for t in trds], fh, indent=2)
print(f"{len(trds)} TRDs bootstrapped — saved to {TRD_PATH}  ({time.time()-t0:.1f}s)")
for t in trds[:5]:
    print(f"  TRD {t.trd_id:2d}: {t.label}  (support={t.support})")

## 9 · Evaluation

- **Parse accuracy** — fraction of held-out nodes where the lexicon's   best-predicted modal mode matches the observed mode
- **TRD coverage** — fraction of held-out graphs successfully assigned to a TRD cluster

In [ ]:
from collections import Counter

parse_acc = inducer.parse_accuracy(held_out_graphs)

# TRD coverage
from trd_bootstrap import graph_to_modal_vector
assigned = 0
for g in held_out_graphs:
    if bootstrapper._model and bootstrapper.vocab:
        vec = graph_to_modal_vector(g, bootstrapper.vocab)
        if any(v > 0 for v in vec):
            assigned += 1
trd_cov = assigned / max(len(held_out_graphs), 1)

print(f"Parse accuracy  : {parse_acc:.1%}")
print(f"TRD coverage    : {trd_cov:.1%}  ({assigned}/{len(held_out_graphs)} graphs)")

## 10 · Distribution Visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Modal mode pie chart
mode_counts = Counter()
for g in train_graphs:
    for e in g.edges:
        mode_counts[e.modal_mode] += 1
if mode_counts:
    axes[0].pie(mode_counts.values(), labels=mode_counts.keys(), autopct="%1.1f%%", startangle=140)
    axes[0].set_title("Modal Mode Distribution")

# UCCA category bar chart
UCCA_NAMES = {0:"Scene",1:"Process",2:"Connector",3:"Ground",4:"Adverbial",5:"State",6:"Participant"}
cat_counts = Counter()
for g in train_graphs:
    for n in g.nodes:
        cat_counts[UCCA_NAMES.get(n.category_id, str(n.category_id))] += 1
if cat_counts:
    axes[1].bar(cat_counts.keys(), cat_counts.values(), color="steelblue")
    axes[1].set_title("UCCA Category Distribution")
    axes[1].set_xlabel("Category")
    axes[1].set_ylabel("Count")
    plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.show()

## 11 · Inference Demo

The trained model can now **analyse any new English sentence**:

1. Preprocess → parse → MTLG graph
2. Look up each lemma in the induced lexicon
3. Project the graph's modal profile → nearest TRD centroid

In [ ]:
import numpy as np
from trd_bootstrap import graph_to_modal_vector

MODE_SYM = {"diamond": "◇", "box": "□", "lozenge": "◊"}

def analyse(sentence: str):
    tokens = preprocess_for_type_assignment(sentence, LANGUAGE)
    clean  = " ".join(t.split("[")[0] for t in tokens)
    tree   = parser.parse(clean)
    g      = ud_tree_to_mtlg(tree)

    # Lexicon lookup per node
    rows = []
    for node in g.nodes:
        entry = lexicon.best_type(node.lemma)
        rows.append({
            "token"  : node.text,
            "upos"   : node.upos,
            "mode"   : MODE_SYM.get(entry.modal_mode, "?") if entry else "—",
            "ucca"   : UCCA_NAMES.get(node.category_id, str(node.category_id)),
            "arity"  : node.arity,
            "count"  : int(entry.count) if entry else 0,
        })

    # TRD assignment
    trd_label = "unknown"
    if bootstrapper._model and bootstrapper.vocab:
        vec    = graph_to_modal_vector(g, bootstrapper.vocab)
        trd_id = bootstrapper._model.predict(vec)
        for t in trds:
            if t.trd_id == trd_id:
                trd_label = t.label
                break

    return rows, trd_label, g

print("analyse() ready.")

In [ ]:
demo_sentences = [
    "The scientist discovered a new particle in the laboratory.",
    "Climate change is accelerating global warming.",
    "She quickly solved the complex mathematical equation.",
]

for sent in demo_sentences:
    rows, trd_label, _ = analyse(sent)
    print(f"\nSentence : {sent}")
    print(f"TRD      : {trd_label}")
    print(f"{'Token':<18} {'UPOS':<8} {'Mode':<4} {'UCCA':<12} {'Arity':<6} {'Count'}")
    print("-" * 60)
    for r in rows:
        print(f"{r['token']:<18} {r['upos']:<8} {r['mode']:<4} {r['ucca']:<12} {r['arity']:<6} {r['count']}")

## 12 · MTLG Graph Visualisation

NetworkX renders the modal dependency graph for the first demo sentence.
Node colour = UCCA category; edge style = modal mode (◇ solid, □ dashed, ◊ dotted).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

TARGET_SENT = demo_sentences[0]
_, _, g = analyse(TARGET_SENT)

UCCA_COLORS = {
    1: "#DD8452", 6: "#4C72B0", 0: "#55A868",
    5: "#C44E52", 4: "#8172B3",
    2: "#937860", 3: "#DA8BC3",
}
MODE_STYLE = {"diamond": "solid", "box": "dashed", "lozenge": "dotted"}
MODE_COLOR = {"diamond": "#4C72B0", "box": "#DD8452", "lozenge": "#55A868"}
MODE_SYM   = {"diamond": "◇", "box": "□", "lozenge": "◊"}

G = nx.DiGraph()
for node in g.nodes:
    G.add_node(node.token_id, label=node.text, ucca=node.category_id)
for edge in g.edges:
    G.add_edge(edge.src_id, edge.dst_id, mode=edge.modal_mode, deprel=edge.deprel)

node_colors = [UCCA_COLORS.get(G.nodes[n].get("ucca", 0), "#aaaaaa") for n in G.nodes]
pos = nx.spring_layout(G, seed=42, k=2.2)

fig, ax = plt.subplots(figsize=(14, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1000, ax=ax, alpha=0.95)
nx.draw_networkx_labels(G, pos,
    labels={n: G.nodes[n].get("label", str(n)) for n in G.nodes},
    font_size=8, font_color="white", font_weight="bold", ax=ax)

for (u, v, data) in G.edges(data=True):
    mode = data.get("mode", "diamond")
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)],
        style=MODE_STYLE.get(mode, "solid"),
        edge_color=MODE_COLOR.get(mode, "#888"),
        arrows=True, arrowsize=20, width=2, ax=ax,
        connectionstyle="arc3,rad=0.1")

edge_labels = {(e.src_id, e.dst_id): e.deprel for e in g.edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)

legend_patches = [mpatches.Patch(color=c, label=UCCA_NAMES[k]) for k, c in UCCA_COLORS.items()]
mode_lines = [
    plt.Line2D([0],[0], color=MODE_COLOR[m], lw=2,
               linestyle=MODE_STYLE[m], label=f"{MODE_SYM[m]} {m}")
    for m in ("diamond","box","lozenge")
]
ax.legend(handles=legend_patches + mode_lines, loc="lower left", fontsize=8,
          title="UCCA / Mode", ncol=2)
ax.set_title(f'MTLG Graph: "{TARGET_SENT}"', fontsize=11, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

## 13 · Download Trained Artefacts

In [ ]:
# Download lexicon and TRD files from Colab to your machine
try:
    from google.colab import files
    files.download(LEXICON_PATH)
    files.download(TRD_PATH)
    print(f"Downloaded {LEXICON_PATH} and {TRD_PATH}")
except ImportError:
    print(f"Artefacts saved locally: {LEXICON_PATH}, {TRD_PATH}")
    print("(Not running in Colab — copy these files from the current directory.)")

## 14 · Launch Streamlit UI

The repository includes a full **Streamlit web interface** (`app.py`) for interactive training and inference with real-time MTLG graph visualisation.

Run the cell below to start the UI — a public URL will be printed via **pyngrok**.

In [ ]:
import subprocess, threading, time, os

APP_PATH = os.path.join(REPO_ROOT, "app.py")

# Start Streamlit in the background
_proc = subprocess.Popen(
    ["streamlit", "run", APP_PATH, "--server.port", "8501",
     "--server.headless", "true", "--server.enableCORS", "false"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(3)  # give Streamlit a moment to start

# Expose via ngrok
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(8501)
    print(f"Streamlit UI is live at: {public_url}")
except Exception as e:
    print(f"Streamlit started (PID {_proc.pid}). Could not open ngrok tunnel: {e}")
    print("If running locally, visit: http://localhost:8501")